In [13]:
import os
import cv2
import numpy as np
import pandas as pd
import random

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from sklearn.model_selection import train_test_split

from tqdm import tqdm
import matplotlib.pyplot as plt

In [2]:
import pandas as pd
import os

dataset = []

with open("../Datasets/iam_words/words.txt", "r") as f:
    for line in f:
        if line.startswith("#"):
            continue

        parts = line.strip().split()

        if len(parts) < 9:
            continue

        word_id = parts[0]
        status = parts[1]

        if status != "ok":
            continue

        label = " ".join(parts[8:])

        folder1 = word_id.split("-")[0]
        folder2 = "-".join(word_id.split("-")[:2])

        image_path = f"../Datasets/iam_words/words/{folder1}/{folder2}/{word_id}.png"

        if os.path.exists(image_path):
            dataset.append([image_path, label])

df = pd.DataFrame(dataset, columns=["image_path", "label"])

print(df.head())
print("Total samples:", len(df))

                                          image_path label
0  ../Datasets/iam_words/words/a01/a01-000u/a01-0...     A
1  ../Datasets/iam_words/words/a01/a01-000u/a01-0...  MOVE
2  ../Datasets/iam_words/words/a01/a01-000u/a01-0...    to
3  ../Datasets/iam_words/words/a01/a01-000u/a01-0...  stop
4  ../Datasets/iam_words/words/a01/a01-000u/a01-0...   Mr.
Total samples: 38305


In [3]:
characters = set()

for text in df["label"]:
    characters.update(text)

vocab = sorted(list(characters))

print(vocab)
print("Vocabulary Size:", len(vocab))

[' ', '!', '"', '#', "'", '(', ')', '*', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
Vocabulary Size: 77


In [8]:
IMAGE_SIZE = (128, 32)
BATCH_SIZE = 64
EPOCHS = 50
PADDING_TOKEN = 99

In [4]:
import os
import tensorflow as tf

DATA_INPUT_PATH = "../Datasets/iam_words"

images_path = []
labels = []

def preprocess_dataset():

    characters = set()
    max_len = 0

    with open(
        os.path.join(DATA_INPUT_PATH, "words.txt"),
        "r"
    ) as file:

        for line in file:

            if line.startswith("#"):
                continue

            parts = line.strip().split()

            if len(parts) < 9:
                continue

            word_id = parts[0]
            status = parts[1]

            # Only keep valid samples
            if status != "ok":
                continue

            folder1 = word_id.split("-")[0]
            folder2 = "-".join(word_id.split("-")[:2])

            image_path = os.path.join(
                DATA_INPUT_PATH,
                "words",
                folder1,
                folder2,
                f"{word_id}.png"
            )

            # Skip missing images
            if not os.path.isfile(image_path):
                continue

            label = " ".join(parts[8:])

            images_path.append(image_path)
            labels.append(label)

            for char in label:
                characters.add(char)

            max_len = max(
                max_len,
                len(label)
            )

    characters = sorted(list(characters))

    print("Total samples:", len(images_path))
    print("Vocabulary size:", len(characters))
    print("Max label length:", max_len)

    char_to_num = tf.keras.layers.StringLookup(
        vocabulary=characters,
        mask_token=None
    )

    num_to_char = tf.keras.layers.StringLookup(
        vocabulary=char_to_num.get_vocabulary(),
        mask_token=None,
        invert=True
    )

    return (
        characters,
        char_to_num,
        num_to_char,
        max_len
    )

characters, char_to_num, num_to_char, max_len = preprocess_dataset()

Total samples: 38305
Vocabulary size: 77
Max label length: 19


In [9]:
def distortion_free_resize(image, img_size):
    w, h = img_size
    image = tf.image.resize(image, size=(h, w), preserve_aspect_ratio=True)

    # Check tha amount of padding needed to be done.
    pad_height = h - tf.shape(image)[0]
    pad_width = w - tf.shape(image)[1]

    # Only necessary if you want to do same amount of padding on both sides.
    if pad_height % 2 != 0:
        height = pad_height // 2
        pad_height_top = height + 1
        pad_height_bottom = height
    else:
        pad_height_top = pad_height_bottom = pad_height // 2

    if pad_width % 2 != 0:
        width = pad_width // 2
        pad_width_left = width + 1
        pad_width_right = width
    else:
        pad_width_left = pad_width_right = pad_width // 2

    image = tf.pad(
        image,
        paddings=[
            [pad_height_top, pad_height_bottom],
            [pad_width_left, pad_width_right],
            [0, 0],
        ],
    )

    image = tf.transpose(image, perm=[1, 0, 2])
    image = tf.image.flip_left_right(image)
    return image

def preprocess_image(image_path, img_size):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_png(image, 1)
    image = distortion_free_resize(image, img_size)
    image = tf.cast(image, tf.float32) / 255.0
    return image

def vectorize_label(label):
    label = char_to_num(tf.strings.unicode_split(
        label, input_encoding="UTF-8"))
    length = tf.shape(label)[0]
    pad_amount = max_len - length
    label = tf.pad(label, paddings=[[0, pad_amount]],
                   constant_values=PADDING_TOKEN)
    return label

In [10]:
def process_images_labels(image_path, label):
    image = preprocess_image(image_path, IMAGE_SIZE)
    label = vectorize_label(label)
    return {"image": image, "label": label}

def prepare_dataset(image_paths, labels):
    AUTOTUNE = tf.data.AUTOTUNE
    print('len(image_paths): ', len(image_paths))
    print('len(labels): ', len(labels))
    dataset = tf.data.Dataset.from_tensor_slices((image_paths, labels)).map(
        process_images_labels, num_parallel_calls=AUTOTUNE
    )
    return dataset.batch(BATCH_SIZE).cache().prefetch(AUTOTUNE)

In [11]:
def split_dataset():
    # Split the data into training, validation, and test sets using train_test_split
    train_images, test_images, train_labels, test_labels = train_test_split(
        images_path, labels, test_size=0.2, random_state=42
    )

    # Further split the test set into validation and final test sets
    val_images, test_images, val_labels, test_labels = train_test_split(
        test_images, test_labels, test_size=0.5, random_state=42
    )

    train_set = prepare_dataset(train_images, train_labels)
    val_set = prepare_dataset(val_images, val_labels)
    test_set = prepare_dataset(test_images, test_labels)
    
    return train_set, val_set, test_set

train_set, val_set, test_set = split_dataset()

len(image_paths):  30644
len(labels):  30644
len(image_paths):  3830
len(labels):  3830
len(image_paths):  3831
len(labels):  3831


In [12]:
class CTCLayer(tf.keras.layers.Layer):
    def __init__(self, name=None):
        super().__init__(name=name)
        self.loss_fn = tf.keras.backend.ctc_batch_cost

    def call(self, y_true, y_pred):
        batch_len = tf.cast(tf.shape(y_true)[0], dtype="int64")
        input_length = tf.cast(tf.shape(y_pred)[1], dtype="int64")
        label_length = tf.cast(tf.shape(y_true)[1], dtype="int64")

        input_length = input_length * \
            tf.ones(shape=(batch_len, 1), dtype="int64")
        label_length = label_length * \
            tf.ones(shape=(batch_len, 1), dtype="int64")
        loss = self.loss_fn(y_true, y_pred, input_length, label_length)
        self.add_loss(loss)

        # At test time, just return the computed predictions.
        return y_pred